# Watershed Segmentation for Image Analysis

This notebook demonstrates watershed segmentation on an input image and exports the result as an SVG file with separate paths for each segment.

## Overview

The watershed algorithm is a powerful image segmentation technique that treats the grayscale image as a topographic surface and finds the watershed lines that separate different regions.

### Steps:
1. Load and preprocess the image
2. Apply watershed segmentation
3. Extract contours for each segment
4. Export to SVG with separate paths

In [ ]:
# Import required libraries
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load the Input Image

In [ ]:
# Load image
input_path = 'segment_scan.jpg'
img = cv2.imread(input_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
height, width = img.shape[:2]

# Display original image
plt.imshow(gray, cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.show()

print(f"Image size: {width} x {height}")

## 2. Preprocess the Image

We apply bilateral filtering to reduce noise while preserving edges, then use adaptive thresholding to create a binary image.

In [ ]:
# Denoise using bilateral filter
denoised = cv2.bilateralFilter(gray, 9, 75, 75)

# Apply adaptive thresholding
binary = cv2.adaptiveThreshold(
    denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV, 11, 2
)

# Morphological opening to remove noise
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)

# Display preprocessing results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(denoised, cmap='gray')
axes[0].set_title('Denoised')
axes[0].axis('off')

axes[1].imshow(binary, cmap='gray')
axes[1].set_title('Binary (Adaptive Threshold)')
axes[1].axis('off')

axes[2].imshow(opening, cmap='gray')
axes[2].set_title('After Morphological Opening')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Apply Watershed Segmentation

The watershed algorithm uses distance transform to identify regions and then segments the image.

In [ ]:
# Distance transform
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)

# Threshold to get sure foreground
_, sure_fg = cv2.threshold(dist_transform, 0.3 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# Dilate to get sure background
kernel = np.ones((3, 3), np.uint8)
sure_bg = cv2.dilate(opening, kernel, iterations=3)

# Find unknown region
unknown = cv2.subtract(sure_bg, sure_fg)

# Display intermediate results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(dist_transform, cmap='jet')
axes[0, 0].set_title('Distance Transform')
axes[0, 0].axis('off')

axes[0, 1].imshow(sure_fg, cmap='gray')
axes[0, 1].set_title('Sure Foreground')
axes[0, 1].axis('off')

axes[1, 0].imshow(sure_bg, cmap='gray')
axes[1, 0].set_title('Sure Background')
axes[1, 0].axis('off')

axes[1, 1].imshow(unknown, cmap='gray')
axes[1, 1].set_title('Unknown Region')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Label markers for watershed
_, markers = cv2.connectedComponents(sure_fg)

# Add 1 to all labels so background is not 0, but 1
markers = markers + 1

# Mark unknown region as 0
markers[unknown == 255] = 0

# Create 3-channel image for watershed
img_for_watershed = cv2.merge([opening, opening, opening])

# Apply watershed
labels = cv2.watershed(img_for_watershed, markers)

# Visualize segmentation result
# Create a colored overlay for visualization
overlay = img_rgb.copy()
overlay[labels == -1] = [255, 0, 0]  # Mark boundaries in red

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(labels, cmap='nipy_spectral')
axes[0].set_title('Watershed Segmentation Labels')
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title('Segmentation Boundaries')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Count segments
unique_labels = np.unique(labels)
unique_labels = unique_labels[(unique_labels > 1)]
print(f"Number of segments detected: {len(unique_labels)}")

## 4. Extract Contours from Segments

In [ ]:
def extract_contours(labels, min_area=50):
    """
    Extract contours from segmented labels.
    
    Args:
        labels: Watershed labels array
        min_area: Minimum area threshold to filter small segments
    
    Returns:
        List of contours (each contour is a list of (x, y) points)
    """
    contours_list = []
    
    # Get unique labels (excluding background -1 and 1)
    unique_labels = np.unique(labels)
    unique_labels = unique_labels[(unique_labels > 1)]
    
    for label in unique_labels:
        # Create binary mask for this label
        mask = np.uint8(labels == label) * 255
        
        # Find contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for contour in contours:
            # Calculate area and filter small regions
            area = cv2.contourArea(contour)
            if area >= min_area:
                # Simplify contour to reduce points
                epsilon = 0.005 * cv2.arcLength(contour, True)
                approx = cv2.approxPolyDP(contour, epsilon, True)
                
                # Convert to list of (x, y) tuples
                points = [(float(pt[0][0]), float(pt[0][1])) for pt in approx]
                if len(points) >= 3:  # Need at least 3 points for a polygon
                    contours_list.append(points)
    
    return contours_list

# Extract contours
contours = extract_contours(labels, min_area=50)
print(f"Total contours extracted: {len(contours)}")

# Visualize contours on original image
contour_img = img_rgb.copy()
for i, contour in enumerate(contours):
    pts = np.array(contour, dtype=np.int32)
    cv2.polylines(contour_img, [pts], True, (255, 0, 0), 2)

plt.figure(figsize=(12, 8))
plt.imshow(contour_img)
plt.title(f'Detected Contours ({len(contours)} segments)')
plt.axis('off')
plt.show()

## 5. Export to SVG

Each segment is exported as a separate SVG path element.

In [ ]:
def path_d_from_points(pts):
    """
    Convert list of points to SVG path 'd' attribute.
    """
    if not pts or len(pts) < 3:
        return ""
    
    # Start with Move command
    path_parts = [f"M {pts[0][0]:.2f},{pts[0][1]:.2f}"]
    
    # Add Line commands for remaining points
    for x, y in pts[1:]:
        path_parts.append(f"L {x:.2f},{y:.2f}")
    
    # Close path
    path_parts.append("Z")
    
    return " ".join(path_parts)


def export_svg(contours, width, height, output_path):
    """
    Export contours as SVG file with separate paths for each segment.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, "w") as f:
        # Write SVG header
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write(f'<svg xmlns="http://www.w3.org/2000/svg" ')
        f.write(f'width="{width}" height="{height}" ')
        f.write(f'viewBox="0 0 {width} {height}">\n')
        
        # Write group for all segments
        f.write('  <g id="watershed-segments" fill="none" stroke="#ff0000" stroke-width="2">\n')
        
        # Write each contour as a separate path
        for i, contour in enumerate(contours):
            path_d = path_d_from_points(contour)
            if path_d:
                f.write(f'    <path id="segment_{i}" d="{path_d}" />\n')
        
        f.write("  </g>\n")
        f.write("</svg>\n")


# Export to SVG
output_svg = 'segment_scan_segmented.svg'
export_svg(contours, width, height, output_svg)
print(f"SVG exported to: {output_svg}")
print(f"Total segments in SVG: {len(contours)}")

## Summary

This notebook demonstrates a complete watershed segmentation pipeline:

1. **Image Loading**: Loaded the input image and converted to grayscale
2. **Preprocessing**: Applied bilateral filtering, adaptive thresholding, and morphological operations
3. **Watershed Segmentation**: Used distance transform and watershed algorithm to segment the image
4. **Contour Extraction**: Extracted and simplified contours for each segment
5. **SVG Export**: Exported each segment as a separate SVG path

The output SVG file contains individual paths for each detected segment, which can be imported into vector graphics software like GIMP, Inkscape, or Adobe Illustrator.